# Smart Feature Ablation Study (Chronos-2)

This notebook implements a **Smart Feature Selection** strategy to identify the most predictive features for stock volatility.

**Process per Stock:**
1.  **Baseline**: Evaluate model performance with **NO** covariates.
2.  **Screening**: Test every available feature **individually**. 
3.  **Ranking**: Rank features by performance improvement.
4.  **Optimization**: Test combinations (Top-K) to find the sweet spot.
5.  **Visualization**: Generate forecast plots for the **BEST** model found.

**Key Features:**
- **GPU Acceleration**: Uses `cuda` if available.
- **Dynamic Features**: Automatically detects ALL numeric features.
- **Visual Verification**: Plots forecasts with confidence intervals for every stock.

In [ ]:
# Core Imports
import os
import glob
import time
import pandas as pd
import numpy as np
import torch
from chronos import Chronos2Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configuration
DATA_DIR = "processed_data"
RESULTS_FILE = "smart_ablation_results.csv"
MODEL_NAME = "amazon/chronos-2" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_COL = "ctc_vol"
DATE_COL = "timestamp"
ID_COL = "id"
DEFAULT_ID = "series_1"

# Training parameters
SPLIT_START_DATE = "2020-01-01"
TRAIN_FRACTION = 0.80
PREDICTION_LENGTH = 1 # Next day prediction
FREQ = "B" # Business days

print(f"Using device: {DEVICE}")

In [ ]:
# Helper Functions

def get_all_features(df, exclude_cols):
    """Dynamically identify all numeric feature columns."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    features = [c for c in numeric_cols if c not in exclude_cols]
    return features

def load_and_prep_data(filepath):
    """Load CSV, reindex to business days, and handle missing values."""
    try:
        df = pd.read_csv(filepath)
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])
        df = df.sort_values(DATE_COL).reset_index(drop=True)
        
        # Add ID column if missing
        if ID_COL not in df.columns:
            df[ID_COL] = DEFAULT_ID
            
        # Business day reindexing
        pieces = []
        for sid, g in df.groupby(ID_COL):
            g = g.sort_values(DATE_COL).drop_duplicates(subset=[DATE_COL])
            g = g.set_index(DATE_COL)
            
            # Create full business day range
            start_date = g.index.min()
            end_date = g.index.max()
            bdays_idx = pd.date_range(start=start_date, end=end_date, freq=FREQ)
            
            # Reindex and forward fill
            g = g.reindex(bdays_idx)
            g = g.ffill() # Forward fill missing values from reindexing
            
            g[ID_COL] = sid
            g.index.name = DATE_COL
            pieces.append(g.reset_index())
            
        df_processed = pd.concat(pieces, ignore_index=True)
        
        # Filter for study period
        df_processed = df_processed[df_processed[DATE_COL] >= pd.Timestamp(SPLIT_START_DATE)]
        
        return df_processed.dropna() 
        
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def calculate_metrics(y_true, y_pred):
    """Calculate deterministic metrics."""
    metrics = {}
    metrics['mae'] = mean_absolute_error(y_true, y_pred)
    metrics['mse'] = mean_squared_error(y_true, y_pred)
    metrics['rmse'] = np.sqrt(metrics['mse'])
    metrics['r2'] = r2_score(y_true, y_pred)
    return metrics

In [ ]:
# Initialize Model
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_NAME,
    device_map=DEVICE,
    torch_dtype=torch.bfloat16,
)
print("Model loaded successfully.")

In [ ]:
# Run Smart Ablation Study

results = []
csv_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
print(f"Found {len(csv_files)} stock files.")

for filepath in csv_files:
    stock_name = os.path.basename(filepath).replace("data_", "").replace(".csv", "")
    print(f"\n{'='*50}")
    print(f"Processing {stock_name}...")
    print(f"{'='*50}")
    
    # Load data
    df = load_and_prep_data(filepath)
    if df is None or len(df) < 50:
        print(f"Skipping {stock_name} due to insufficient data.")
        continue
        
    # Identify ALL features
    exclude = [TARGET_COL, DATE_COL, ID_COL]
    all_features = get_all_features(df, exclude)
    print(f"Found {len(all_features)} potential features.")
    
    # Train/Test Split
    n_total = len(df)
    n_train = int(np.floor(TRAIN_FRACTION * n_total))
    
    context_df = df.iloc[:n_train].copy()
    future_df = df.iloc[n_train:].copy()
    actuals = future_df[TARGET_COL].values
    
    # ---------------------------------------------------------
    # 1. Baseline (No Features)
    # ---------------------------------------------------------
    print("Running Baseline (No Features)...")
    try:
        start_time = time.time()
        # Context needs target only
        context_input = context_df[[ID_COL, DATE_COL, TARGET_COL]]
        
        forecast = pipeline.predict_df(
            context_input,
            prediction_length=len(future_df),
            quantile_levels=[0.5],
            target=TARGET_COL,
            id_column=ID_COL,
            timestamp_column=DATE_COL
        )
        preds = forecast['0.5'].values
        metrics = calculate_metrics(actuals, preds)
        
        baseline_result = {
            'stock': stock_name,
            'features': 'BASELINE_NONE',
            'n_features': 0,
            'mae': metrics['mae'],
            'mse': metrics['mse'],
            'rmse': metrics['rmse'],
            'r2': metrics['r2'],
            'time': time.time() - start_time
        }
        results.append(baseline_result)
        print(f"  Baseline MAE: {metrics['mae']:.6f}")
        
    except Exception as e:
        print(f"  Error in Baseline: {e}")

    # ---------------------------------------------------------
    # 2. Individual Feature Screening
    # ---------------------------------------------------------
    print(f"Screening {len(all_features)} features individually...")
    feature_performance = []
    
    for i, feat in enumerate(all_features):
        if i % 10 == 0: print(f"  Feature {i}/{len(all_features)}", end="\r")
        
        try:
            start_time = time.time()
            feats = [feat]
            
            context_cols = [ID_COL, DATE_COL, TARGET_COL] + feats
            future_cols = [ID_COL, DATE_COL] + feats
            
            forecast = pipeline.predict_df(
                context_df[context_cols],
                future_df[future_cols],
                prediction_length=len(future_df),
                quantile_levels=[0.5],
                target=TARGET_COL,
                id_column=ID_COL,
                timestamp_column=DATE_COL
            )
            preds = forecast['0.5'].values
            metrics = calculate_metrics(actuals, preds)
            
            res = {
                'stock': stock_name,
                'features': feat,
                'n_features': 1,
                'mae': metrics['mae'],
                'mse': metrics['mse'],
                'rmse': metrics['rmse'],
                'r2': metrics['r2'],
                'time': time.time() - start_time
            }
            results.append(res)
            feature_performance.append((feat, metrics['mae']))
            
        except Exception as e:
            print(f"  Error with feature {feat}: {e}")
            
    # ---------------------------------------------------------
    # 3. Top-K Sweep (Finding the Optimal K)
    # ---------------------------------------------------------
    # Sort features by MAE (ascending -> lower is better)
    feature_performance.sort(key=lambda x: x[1])
    
    # Test a range of K to find the "sweet spot"
    k_values = [1, 3, 5, 7, 10, 12, 15, 20]
    
    # Also test ALL features
    combinations_to_test = [(f"TOP_{k}", k) for k in k_values]
    combinations_to_test.append(("ALL_FEATURES", len(all_features)))
    
    best_mae = float('inf')
    best_k_name = "None"
    best_k_val = 0
    
    for name, k in combinations_to_test:
        try:
            if k > len(all_features): k = len(all_features)
            
            # Select features
            if name == "ALL_FEATURES":
                selected_feats = all_features
            else:
                selected_feats = [x[0] for x in feature_performance[:k]]
            
            # Skip if we already tested this exact set (e.g. if len(all) < 20)
            if len(selected_feats) == 0: continue
            
            print(f"Testing combination: {name} ({len(selected_feats)} features)...")
            start_time = time.time()
            
            context_cols = [ID_COL, DATE_COL, TARGET_COL] + selected_feats
            future_cols = [ID_COL, DATE_COL] + selected_feats
            
            forecast = pipeline.predict_df(
                context_df[context_cols],
                future_df[future_cols],
                prediction_length=len(future_df),
                quantile_levels=[0.5],
                target=TARGET_COL,
                id_column=ID_COL,
                timestamp_column=DATE_COL
            )
            preds = forecast['0.5'].values
            metrics = calculate_metrics(actuals, preds)
            
            combo_res = {
                'stock': stock_name,
                'features': f"{name}_COMBINED",
                'n_features': len(selected_feats),
                'mae': metrics['mae'],
                'mse': metrics['mse'],
                'rmse': metrics['rmse'],
                'r2': metrics['r2'],
                'time': time.time() - start_time
            }
            results.append(combo_res)
            print(f"  {name} MAE: {metrics['mae']:.6f}")
            
            # Track best model
            if metrics['mae'] < best_mae:
                best_mae = metrics['mae']
                best_k_name = name
                best_k_val = len(selected_feats)
            
        except Exception as e:
            print(f"  Error with {name}: {e}")

    # ---------------------------------------------------------
    # 4. Visualization (BEST Forecast)
    # ---------------------------------------------------------
    try:
        print(f"Generating forecast plot for {stock_name} using BEST model: {best_k_name} ({best_k_val} features)...")
        
        # Use Best-K features for plotting
        if best_k_name == "ALL_FEATURES":
            plot_feats = all_features
        else:
            # Extract K from name or use best_k_val
            plot_feats = [x[0] for x in feature_performance[:best_k_val]]
        
        context_cols = [ID_COL, DATE_COL, TARGET_COL] + plot_feats
        future_cols = [ID_COL, DATE_COL] + plot_feats
        
        # Generate forecast with quantiles for plotting
        forecast = pipeline.predict_df(
            context_df[context_cols],
            future_df[future_cols],
            prediction_length=len(future_df),
            quantile_levels=[0.1, 0.5, 0.9],
            target=TARGET_COL,
            id_column=ID_COL,
            timestamp_column=DATE_COL
        )
        
        # Plotting
        plt.figure(figsize=(15, 7))
        
        # Plot historical context (last 100 days)
        plt.plot(context_df[DATE_COL].iloc[-100:], context_df[TARGET_COL].iloc[-100:], color="black", label="History")
        
        # Plot Actual Future
        plt.plot(future_df[DATE_COL], future_df[TARGET_COL], color="green", label="Actual")
        
        # Plot Forecast
        plt.plot(forecast[DATE_COL], forecast['0.5'], color="blue", label="Forecast (Median)")
        plt.fill_between(forecast[DATE_COL], forecast['0.1'], forecast['0.9'], color="blue", alpha=0.2, label="80% Confidence Interval")
        
        plt.title(f"Chronos-2 Forecast: {stock_name} (Best Model: {best_k_name})")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show() # Display inline
        
    except Exception as e:
        print(f"Error plotting {stock_name}: {e}")

    # Save intermediate results
    pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
    print(f"Saved results for {stock_name}")

print("\nSmart Ablation study complete!")